# Lab 3: Sieci neuronowe i Deep Learning

## Wstęp

Na tym laboratorium poznamy podstawy sieci neuronowych i uczenia głębokiego (*deep learning*). Najpierw
zaimplementujemy od zera w NumPy regresję logistyczną i jej trening spadkiem wzdłuż gradientu z propagacją wsteczną.
Następnie przeniesiemy to samo do frameworka PyTorch (tensory, autograd, `nn.Module`, optymalizatory), a na koniec
zbudujemy sieć MLP, która poradzi sobie z danymi nieseparowalnymi liniowo.

**Uwagi:**
- pod zadaniami znajdują się komórki sprawdzające (`check_task_...`), które porównują Twoje rozwiązanie
  z implementacją referencyjną i podpowiadają, co poprawić,
- w szablonach usuń linię `raise NotImplementedError` i uzupełnij miejsca oznaczone `...`,
- niektóre zadania wymagają pisemnej odpowiedzi w miejscu oznaczonym `// skomentuj tutaj`,
- uruchamiaj notebook po kolei - dalsze komórki korzystają z Twoich wcześniejszych rozwiązań.

## Wykorzystywane biblioteki

* [NumPy](https://numpy.org/) - obliczenia macierzowe; w nim napiszemy regresję logistyczną od zera
* [PyTorch](https://pytorch.org/) - framework do uczenia głębokiego: tensory, automatyczne różniczkowanie, warstwy sieci i optymalizatory
* [scikit-learn](https://scikit-learn.org/stable/) - tutaj tylko do wygenerowania zbioru danych `make_moons`
* [matplotlib](https://matplotlib.org/) - wykresy i wizualizacje

## Konfiguracja środowiska

Możesz korzystać ze środowiska lokalnego z menedżerem pakietów [uv](https://docs.astral.sh/uv/) lub z [Google Colab](https://colab.research.google.com/). Wszystko zostało przetestowane z użyciem `uv` - jeżeli w trakcie pracy instalujesz biblioteki ręcznie przez `pip`, to coś może się zepsuć lub nie działać z powodu niekompatybilnych wersji.

### Środowisko lokalne

Konfiguracja wygląda tak samo, jak w laboratoriach 1 i 2:
1. Instalacja uv [zgodnie z dokumentacją](https://docs.astral.sh/uv/getting-started/installation/) (jeżeli nie masz go jeszcze zainstalowanego)
2. Stworzenie środowiska wirtualnego: `uv venv`
3. Aktywacja środowiska wirtualnego: zgodnie z komendą wypisaną przez `uv venv`, np. `source .venv/bin/activate` na Linuxie/macOS/WSL
4. Zainstalowanie zależności z pliku `uv.lock`: `uv sync`
5. Uruchomienie Jupyter Notebooka: `uv run jupyter notebook` (samo `jupyter notebook` też zadziała w aktywowanym venv'ie)

Instalowana jest wersja PyTorcha na CPU - w tym laboratorium GPU nie jest potrzebne.

### Google Colab

Jeżeli korzystasz z tej opcji, to prześlij do katalogu `/content` plik `checkers.py` z funkcjami sprawdzającymi oraz plik `requirements.txt`. PyTorch jest w Colabie zainstalowany domyślnie, a pozostałe zależności zainstalujesz, wykonując `!pip install -r requirements.txt` w osobnej komórce notebooka. Schematy z katalogu `figures/` nie wyświetlą się w Colabie, ale nie są potrzebne do rozwiązania zadań.

In [ ]:
# uncomment and run if using Google Colab
# !pip install -r requirements.txt

### Podstawowe importy

Funkcje sprawdzające importujemy z pliku `checkers.py`, który musi leżeć obok notebooka.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from checkers import *

np.set_printoptions(precision=4, suppress=True)

---

Na dotychczasowych laboratoriach zarówno do definicji modelu, jak i doboru parametrów wykorzystywaliśmy wysokopoziomowe interfejsy scikit-learn. Na tym laboratorium przyjrzymy się w jaki sposób sieci neuronowe oraz inne modele parametryczne mogą zostać zaimplementowane z wykorzystaniem bardziej niskopoziomowego API.


## Regresja logistyczna z użyciem propagacji wstecznej w NumPy

Aby nabyć intuicję dotyczącą tego, w jaki sposób działa propagacja wsteczna i później sieci neuronowe, rozpoczniemy od implementacji regresji logistycznej w bibliotece NumPy.

Jako zbiór danych mamy punkty na płaszczyźnie, opisane cechami $x = (x_1, x_2)$, z dwóch klas ($y \in \{0, 1\}$). W naszym przypadku klasy da się
idealnie rozdzielić prostą (są **liniowo separowalne**). 

Do stworzenia modelu przypisującego klasy każdemu z naszych punktów użyjemy **regresji logistycznej** (znanej z laboratorium 2):
$$
\hat p = \sigma(w^\top x + b) = \frac{1}{1 + e^{-(w^\top x + b)}}
$$

<img src="figures/granica_decyzyjna.png" width="760" alt="granica decyzyjna">

Do treningu naszego modelu używającego regresji logistycznej zaimplementujemy prosty wariant **algorytmu propagacji wstecznej (Backpropagation)**. Składa się on z czterech kroków:



1. **Przejście w przód (forward pass)** - liczymy predykcje $\hat p$ dla aktualnych parametrów $\theta$, rozpoczynając od małych, losowo dobranych wartości. (Zadanie 2)
2. **Obliczenie funkcji kosztu** - mierzymy błąd predykcji $L$. (Zadanie 3)
3. **Przejście wstecz (backward pass)** - liczymy gradient $\nabla_\theta L$, który ustali w jaki sposób mamy aktualizować wagi modelu. (Zadanie 4)
4. **Aktualizacja parametrów** - za pomocą optymalizatora (u nas spadek wzdłuż gradientu) aktualizujemy parametry zgodnie z kierunkiem przeciwnym do wskazywanego przez gradient i krokiem przeskalowanym przez **stałą uczącą (learning rate)**: $\theta \leftarrow \theta - \eta \nabla_\theta L$. (Zadanie 5)

<img src="figures/petla_uczenia.png" width="800" alt="pętla uczenia">

Poniżej definiujemy funkcje pomocnicze rysujące zbiór danych (`plot_dataset`), granicę decyzyjną modelu (`plot_decision_boundary`) oraz
wynik treningu (`plot_training`). Model przekazujemy do nich jako funkcję `predict_proba`, więc zadziałają zarówno
dla modeli w NumPy, jak i tych, które zdefiniujemy później w PyTorchu.

In [ ]:
def make_linear_dataset(n=200, seed=0):
    # points from [-3, 3]^2 labelled by the side of the line 1.5 x1 - x2 + 0.5 = 0;
    # points too close to the line are dropped, so the classes are clearly separated
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3, 3, size=(3 * n, 2))
    z = X @ np.array([1.5, -1.0]) + 0.5

    keep = np.abs(z) > 0.6
    X, z = X[keep][:n], z[keep][:n]
    return X, (z > 0).astype(float)


def accuracy(predict_proba, X, y):
    return np.mean((predict_proba(X) >= 0.5) == y)


def plot_dataset(X, y, ax=None, title=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))

    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=10, edgecolors="k", linewidths=0.3)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title or "")
    return ax


def plot_decision_boundary(predict_proba, X, y, ax=None, title=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))

    # model predictions on a dense grid of points covering the data
    x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
    x2 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200)
    xx1, xx2 = np.meshgrid(x1, x2)
    probs = predict_proba(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)

    # background: probability of class 1, black line: decision boundary (p_hat = 0.5)
    ax.contourf(xx1, xx2, probs, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.4)
    if probs.min() < 0.5 < probs.max():
        ax.contour(xx1, xx2, probs, levels=[0.5], colors="black", linewidths=2)

    return plot_dataset(X, y, ax=ax, title=title)


def plot_training(history, predict_proba, X, y, title="", compare=None, mark_errors=False):
    # compare: optional {label: history} of other training runs, drawn on the same learning curve plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    # left: learning curve
    for label, other_history in (compare or {}).items():
        ax1.plot(other_history, label=label, alpha=0.7)
    ax1.plot(history, label=title or "training", color="black")
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("cost")
    ax1.set_title(f"Learning curve - final cost {history[-1]:.3f}")
    ax1.grid(alpha=0.3)
    ax1.legend()

    # right: decision boundary of the trained model
    plot_decision_boundary(predict_proba, X, y, ax=ax2, title=f"{title} - accuracy {accuracy(predict_proba, X, y):.1%}")
    if mark_errors:
        wrong = (predict_proba(X) >= 0.5) != y
        ax2.scatter(X[wrong, 0], X[wrong, 1], marker="x", color="black", s=20, label="wrong predictions")
        ax2.legend(loc="lower left", fontsize=8)

    fig.tight_layout()
    plt.show()

In [ ]:
X_lin, y_lin = make_linear_dataset()
print("X_lin:", X_lin.shape, " y_lin:", y_lin.shape)

ax = plot_dataset(X_lin, y_lin, title="Linearly separable data")
grid_x1 = np.linspace(-3, 3, 10)
ax.plot(grid_x1, 1.5 * grid_x1 + 0.5, "--", color="gray", label="true boundary\n(unknown to the model)")
ax.set_ylim(-3.3, 3.3)
ax.legend(loc="lower right", fontsize=8)
plt.show()

#### Definiowanie pochodnej

Przed przystąpieniem do treningu modelu, stworzymy funkcje pomocnicze do obliczania pochodnych, które później pomogą nam w wyznaczaniu gradientu.

[Pochodna](https://en.wikipedia.org/wiki/Derivative) $f'(x)$ to nachylenie stycznej do wykresu funkcji w punkcie $x$
([animacja: Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Tangent_function_animation.gif)):

<img src="https://upload.wikimedia.org/wikipedia/commons/2/2d/Tangent_function_animation.gif" width="300" alt="pochodna jako nachylenie stycznej">

Można ją policzyć **analitycznie** (na kartce, z reguł różniczkowania) albo
[**numerycznie**](https://en.wikipedia.org/wiki/Numerical_differentiation) - przesuwając $x$ o małe $h$ i mierząc
zmianę wartości funkcji. Użyjemy **różnicy centralnej (central difference)**:
$$
f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}
$$

#### Zadanie 1 (0.5 punktu)

Uzupełnij:
- `analytical_df(x)` - pochodna funkcji $f(x) = 3x^2 - 4x + 5$ wyprowadzona na kartce,
- `df(func, x, h)` - pochodna numeryczna **dowolnej** funkcji `func` w punkcie `x`.

Obie funkcje mają działać także dla tablic NumPy (element po elemencie).

In [ ]:
def f(x):
    # helper function (given)
    return 3 * x**2 - 4 * x + 5

In [ ]:
def analytical_df(x):
    # derivative of f(x) = 3x^2 - 4x + 5
    raise NotImplementedError


def df(func, x, h=1e-5):
    # numerical derivative of func at x using central difference
    raise NotImplementedError

In [ ]:
check_task_1(analytical_df, df)

Porównajmy pochodne analityczne (linia) i numeryczne (punkty) dla naszej funkcji $f$ oraz dla funkcji z animacji:

$$
g(x) = x \sin(x^2) + 1
$$

której pochodna to:

$$
g'(x) = \sin(x^2) + 2x^2 \cos(x^2)
$$

In [ ]:
def g(x):
    return x * np.sin(x**2) + 1


def analytical_dg(x):
    return np.sin(x**2) + 2 * x**2 * np.cos(x**2)


fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, func, dfunc, name, xs in (
    (axes[0], f, analytical_df, "f(x) = 3x² − 4x + 5", np.linspace(-1.5, 3, 200)),
    (axes[1], g, analytical_dg, "g(x) = x·sin(x²) + 1", np.linspace(-2, 3, 400)),
):
    ax.plot(xs, func(xs), color="gray", label="function")
    ax.plot(xs, dfunc(xs), lw=2, label="analytical derivative")
    ax.plot(xs[::12], df(func, xs[::12]), "o", ms=4, label="numerical derivative (df)")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(name)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

Zauważmy, że pochodna numeryczna wymaga tylko możliwości policzenia funkcji, ale dla modelu z $P$ parametrami gradient kosztuje
$2P$ obliczeń kosztu. Dlatego w praktyce, tam gdzie to możliwe, używa się pochodnych analitycznych. Pochodne numeryczne służą głównie do
ich **sprawdzania** (*gradient checking*) - my będziemy używać ich dla naszej wygody.

#### Definicja modelu i forward pass


Zdefiniujemy teraz nasz model jako klasę `LogisticRegression`. Obiekt tej klasy przechowuje parametry modelu
(`self.w`, `self.b`) i potrafi policzyć predykcje, czyli wykonać **forward pass** (metoda `forward`).
Z bardzo podobnymi klasami spotkamy się później w PyTorchu - tam również model to klasa z parametrami
i metodą `forward`.

Przypomnijmy wzory. Dla punktu $x$ model liczy najpierw logit, czyli kombinację liniową cech:

$$
z = w^\top x + b = w_1 x_1 + w_2 x_2 + b
$$

Dla wszystkich $N$ punktów naraz (wiersze macierzy $X$) to samo zapisujemy wektorowo:

$$
z = X w + b
$$

Później model przepuszcza logit przez funkcję sigmoidalną $\sigma$ i wyznacza prawdopodobieństwo $\hat p$:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

$$
\hat p = \sigma(z)
$$

#### Zadanie 2 (1 punkt)

Uzupełnij:
- `sigmoid(z)` - element po elemencie,
- `LogisticRegression.__init__` - `self.w`: wektor `(n_features,)` z rozkładu normalnego o odchyleniu 0.1
  (`rng.normal`), `self.b`: `0.0`,
- `LogisticRegression.forward` - prawdopodobieństwa klasy 1 dla wszystkich punktów `X` naraz, kształt `(N,)`
  (`X @ w + b` ma kształt $(N, 2) \cdot (2,) \to (N,)$, a `np.exp` działa element po elemencie).

In [ ]:
def sigmoid(z):
    raise NotImplementedError
    return ...  # 1 / (1 + e^(-z))


class LogisticRegression:
    def __init__(self, n_features, seed=0):
        rng = np.random.default_rng(seed)
        raise NotImplementedError
        self.w = ...  # weights, shape (n_features,): normal distribution, mean 0, std 0.1 (rng.normal)
        self.b = ...  # bias, a single number: 0.0

    def forward(self, X):
        # X: (N, n_features) -> probabilities of class 1, shape (N,)
        raise NotImplementedError
        z = ...       
        return ... 

    def predict(self, X, threshold=0.5):
        # hard 0/1 predictions
        return (self.forward(X) >= threshold).astype(float)

In [ ]:
check_task_2(sigmoid, LogisticRegression)

Zwizualizujmy teraz, jaką granicę decyzyjną wyznacza nasz model z losowo zainicjalizowanymi parametrami.

In [ ]:
model = LogisticRegression(2, seed=1)
print("parameters:  w =", model.w, "  b =", model.b)

acc = np.mean(model.predict(X_lin) == y_lin)
plot_decision_boundary(model.forward, X_lin, y_lin, title=f"Random parameters - accuracy {acc:.0%}")
plt.show()

Model jest w stanie wyznaczyć granicę decyzyjną, ale losowe parametry nie pozwalają mu zrobić tego w oczekiwany sposób. W następnych krokach zdefiniujemy proces ich aktualizacji, aby klasyfikator poprawnie potrafił oddzielić od siebie klasy.

#### Funkcja kosztu

Jako funkcji kosztu w naszym algorytmie użyjemy **binarnej entropii krzyżowej (binary cross-entropy, BCE)**, zdefiniowanej w laboratorium 2:
$$
L = BCE(y, \hat{p}) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]
$$

Koszt pojedynczego punktu, czyli wyraz sumy wraz z minusem, będziemy dalej oznaczać $\ell_i$:

$$
\ell_i = -\left[\, y_i \log(\hat p_i) + (1 - y_i) \log(1 - \hat p_i) \,\right]
\qquad
L = \frac{1}{N} \sum_{i=1}^{N} \ell_i
$$

#### Zadanie 3 (0.5 punktu)

Uzupełnij:
- `bce_per_sample` - koszt każdego punktu $\ell_i$ 
- `bce` - średnia po $\ell_i$ 

In [ ]:
def bce_per_sample(p_hat, y, eps=1e-12):
    # binary cross-entropy for each point separately - (N,)
    p_hat = np.clip(p_hat, eps, 1 - eps)  # protects against log(0)
    raise NotImplementedError


def bce(p_hat, y):
    # mean binary cross-entropy - a single number
    raise NotImplementedError

In [ ]:
check_task_3(bce_per_sample, bce)

Na wykresach poniżej widać koszt punktu oraz jego pochodną po logicie $z$ (liczoną Twoją funkcją `df`)
w porównaniu z MSE.
Gdy model bardzo się myli, pochodna BCE wynosi około $-1$ (silny sygnał do poprawy), a pochodna MSE jest prawie
zerowa - dlatego do klasyfikacji używamy entropii krzyżowej.

In [ ]:
p_grid = np.linspace(0.001, 0.999, 300)
z_grid = np.linspace(-8, 8, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(p_grid, -np.log(p_grid), label="y = 1:  −log(p̂)")
ax1.plot(p_grid, -np.log(1 - p_grid), label="y = 0:  −log(1 − p̂)")
ax1.set_xlabel("prediction p̂")
ax1.set_title("point cost")
ax1.set_ylim(0, 5)
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

ax2.plot(z_grid, df(lambda z: -np.log(sigmoid(z)), z_grid), label="BCE")
ax2.plot(z_grid, df(lambda z: (sigmoid(z) - 1.0) ** 2, z_grid), label="MSE")
ax2.axvspan(-8, 0, color="gray", alpha=0.1)
ax2.text(-4, -0.85, "model is wrong", ha="center", fontsize=9)
ax2.set_xlabel("logit z")
ax2.set_title("derivative of the cost w.r.t. z (point of class 1)")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print(f"cost of the random model: {bce(model.forward(X_lin), y_lin):.4f}  (for p̂ = 0.5 everywhere: ln 2 = {np.log(2):.4f})")

### Gradient i propagacja wsteczna

**Gradient** to wektor pochodnych cząstkowych kosztu po każdym z parametrów - u nas
$\nabla_\theta L = \left[ \frac{\partial L}{\partial w_1}, \frac{\partial L}{\partial w_2}, \frac{\partial L}{\partial b} \right]$.
Wskazuje kierunek, w którym koszt najszybciej **rośnie**, więc parametry przesuwamy w stronę przeciwną
(krok 4 pętli uczącej, Zadanie 5).

Koszt to złożenie prostych funkcji:

$$
z_i = w^\top x_i + b, \qquad \hat p_i = \sigma(z_i), \qquad \ell_i(y_i, \hat p_i)
$$

Z twierdzenia o pochodnej złożenia, znanej też jako **reguła łańcuchowa (chain rule)**:

$$
\frac{\partial \ell_i}{\partial w_j} =
\underbrace{\frac{\partial \ell_i}{\partial \hat p_i}}_{\text{BCE}} \cdot
\underbrace{\frac{\partial \hat p_i}{\partial z_i}}_{\text{sigmoida}} \cdot
\underbrace{\frac{\partial z_i}{\partial w_j}}_{\text{logit}}
$$

**Propagacja wsteczna** liczy te iloczyny od końca, czyli od kosztu w stronę parametrów. Dzięki temu część wspólną
dla wszystkich parametrów, $\frac{\partial \ell_i}{\partial \hat p_i} \cdot \frac{\partial \hat p_i}{\partial z_i}$, liczymy tylko raz.

Koszt batcha (u nas całego zbioru treningowego) to średnia $L = \frac{1}{N} \sum_{i=1}^{N} \ell_i$, a każdy parametr
wpływa na niego przez wszystkie punkty. Najpierw liczymy więc dla każdego punktu, jak koszt zależy od jego logitu:

$$
\delta_i = \frac{\partial L}{\partial z_i} = \frac{1}{N} \cdot \frac{\partial \ell_i}{\partial \hat p_i} \cdot \frac{\partial \hat p_i}{\partial z_i}
$$

Wykorzystamy też fakt, że:

$$
\frac{\partial z_i}{\partial w_j} = x_{ij}
\qquad
\frac{\partial z_i}{\partial b} = 1
$$

Pochodna sumy to suma pochodnych, więc gradient otrzymujemy, sumując po wszystkich punktach:

$$
\frac{\partial L}{\partial w_j} = \sum_{i=1}^{N} \delta_i \, x_{ij}
\qquad
\frac{\partial L}{\partial b} = \sum_{i=1}^{N} \delta_i
$$

W kodzie nie liczymy tego pętlą po punktach, tylko macierzowo. Niech $\delta$, $\frac{\partial \ell}{\partial \hat p}$
i $\frac{\partial \hat p}{\partial z}$ oznaczają wektory o kształcie $(N,)$ z wartościami dla wszystkich punktów,
a $\odot$ - mnożenie element po elemencie. Wtedy:

$$
\delta = \frac{1}{N} \, \frac{\partial \ell}{\partial \hat p} \odot \frac{\partial \hat p}{\partial z}
\qquad
\nabla_w L = X^\top \delta
\qquad
\frac{\partial L}{\partial b} = \sum_{i=1}^{N} \delta_i
$$

Iloczyn $X^\top \delta$ ma kształt $(2, N) \cdot (N,) \to (2,)$, a jego $j$-ty element to dokładnie suma $\sum_{i} \delta_i \, x_{ij}$.

#### Zadanie 4 (1 punkt)

Uzupełnij `backward` - funkcja tylko zwraca gradient `(grad_w, grad_b)`, a parametry zmienimy w pętli uczącej
(Zadanie 5). 

Gradient ma zawsze kształt parametru: 
- `grad_w` - `(2,)`, 
- `grad_b` - liczba.

Przykład ręcznie napisanego przejścia w przód i wstecz w NumPy:
[Learning PyTorch with Examples - Warm-up: numpy](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html#warm-up-numpy).

In [ ]:
def backward(model, X, y):
    # gradient of the mean BCE cost w.r.t. model parameters -> (grad_w, grad_b)
    N = len(y)

    z = X @ model.w + model.b  # (N,)
    p_hat = sigmoid(z)         # (N,)

    point_cost = lambda p: bce_per_sample(p, y)  # for use with df(func, x)

    raise NotImplementedError
    # you may use the df(func, x) to calculate the derivatives
    dloss_dp = ...  
    dp_dz = ...    
    delta = ...     
    grad_w = ...    
    grad_b = ...    
    return grad_w, grad_b

In [ ]:
check_task_4(backward, LogisticRegression)

### Pętla ucząca

Mając już wszystkie potrzebne elementy, możemy zaimplementować **pętlę uczącą (training loop)**, która iteracyjnie
optymalizuje parametry - u nas spadkiem wzdłuż gradientu.

#### Zadanie 5 (0.5 punktu)

Uzupełnij `train`. Zmieniaj parametry w miejscu (przypisz nowe wartości do `model.w` i `model.b`), a do historii dopisuj koszt policzony **przed** aktualizacją, jako liczbę (`float(...)`).
Po sprawdzeniu wytrenujemy model, a `plot_training` narysuje krzywą uczenia z Twojej historii i granicę decyzyjną.
Podobną pętlę w NumPy znajdziesz w tutorialu [Learning PyTorch with Examples - Warm-up: numpy](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html#warm-up-numpy).

In [ ]:
def train(model, X, y, lr=0.5, epochs=100):
    # full-batch gradient descent; modifies the model in place, returns cost history (one value per epoch)
    history = []
    for epoch in range(epochs):
        raise NotImplementedError
        p_hat = ...           # 1. forward pass: probabilities for all points (model.forward)   
        loss = ...            # 2. cost before the update (bce) 
        history.append(float(loss))
        grad_w, grad_b = ...  # 3. gradient of the cost (backward)
        model.w = ...         # 4. update: step against the gradient, scaled by lr
        model.b = ...         #    the same for the bias
    return history

In [ ]:
check_task_5(train, LogisticRegression)

In [ ]:
model = LogisticRegression(2, seed=1)
history = train(model, X_lin, y_lin, lr=0.5, epochs=100)
print(f"learned parameters: w = {model.w}, b = {model.b:.4f}")
plot_training(history, model.forward, X_lin, y_lin, title="NumPy")

Wypróbuj inne wartości `lr` (np. `0.01`, `5`, `50`) i liczby epok - jak zmienia się krzywa uczenia?

---
## PyTorch

Dotychczas do tworzenia pętli uczącej używaliśmy matematycznych definicji napisanych w NumPy. Mimo wszystko, większość tworzenia i szkolenia modeli jest raczej schematyczna, więc do tego typu operacji zwykle będziemy chcieli używać frameworka. Jednym z nich jest PyTorch - udostępniony przez zespół Facebook AI Research (FAIR), jego publiczna wersja pojawiła się w 2017 roku.

PyTorch to w uproszczeniu NumPy + GPU + [automatyczne różniczkowanie (autograd)](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html):
gradienty, które poprzednio liczyliśmy ręcznie, policzy za nas.

Najważniejsze moduły:
- [`torch`](https://docs.pytorch.org/docs/stable/torch.html) - tensory i podstawowe funkcje, np. `torch.tensor()`, `torch.from_numpy()`,
- [`torch.nn`](https://docs.pytorch.org/docs/stable/nn.html) - warstwy i funkcje kosztu, np. `nn.Linear`, `nn.Sigmoid`, `nn.BCELoss`,
- [`torch.optim`](https://docs.pytorch.org/docs/stable/optim.html) - optymalizatory, czyli warianty spadku wzdłuż gradientu.

Pętla ucząca ma te same kroki co nasza implementacja w NumPy:

<img src="figures/petla_uczenia_pytorch.png" width="800" alt="pętla w PyTorchu">

| Krok | Kod w NumPy | PyTorch |
|---|---|---|
| parametry + przejście w przód | `LogisticRegression.forward` | `nn.Module`, `nn.Linear`, `nn.Sigmoid`; wywołanie `model(X)` |
| funkcja kosztu | `bce` | `nn.BCELoss()` |
| gradient (backward pass) | `backward` | `loss.backward()` |
| aktualizacja (optymalizator) | `w - lr * grad_w` | `optimizer.step()` (+ `optimizer.zero_grad()`) |

### Tensory

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

Podstawowym obiektem w PyTorchu jest [`Tensor`](https://docs.pytorch.org/docs/stable/tensors.html), klasa reprezentująca tensory (uogólnienie wektora i macierzy) dowolnego wymiaru.

Dodatkowo niektóre tensory będą miały automatycznie obliczony gradient (o tym w części o autogradzie). Co ważne, tensor jest na pewnym urządzeniu ([`device`](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.device)), CPU lub GPU, a przenosić między nimi trzeba explicite, metodą `.to()`. W tym laboratorium wszystko liczymy na CPU.

API tensorów jest bardzo podobne do NumPy. Dwie pułapki, na które możesz natrafić w trakcie pisania kodu w tym laboratorium:

- **typ liczb** ([`dtype`](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)) - NumPy domyślnie tworzy `float64`, a warstwy PyTorcha
  pracują na `float32`. Dane z NumPy zamieniamy przez `.float()` albo `torch.tensor(..., dtype=torch.float32)`. Dotyczy to
  także etykiet: `nn.BCELoss` wymaga etykiet zmiennoprzecinkowych (`0.0` i `1.0`), a nie całkowitych.
- **shape** - model zwraca kolumnę `(N, 1)`, a etykiety mamy jako wektor `(N,)`. Funkcja kosztu wymaga tego samego
  kształtu, więc etykiety zamieniamy w kolumnę przez `.unsqueeze(1)`
  (w drugą stronę: `.squeeze(1)`).

Uwaga odnośnie pamięci: `torch.from_numpy()`tworzy tensor, który współdzieli pamięć
z tablicą NumPy - jest to bardzo szybkie, ale zmiana jednego obiektu jest widoczna w drugim.
`torch.tensor()`zawsze kopiuje dane. 

In [ ]:
a = np.array([[1.0, 2.0], [3.0, 4.0]])
t = torch.from_numpy(a)  # shares memory with the numpy array
print("tensor from NumPy:", t.dtype, tuple(t.shape), t.device)
a[0, 0] = 100.0
print("after changing a[0, 0]: t[0, 0] =", t[0, 0].item(), "<- shared memory")
a[0, 0] = 1.0

t32 = t.float()
print("after .float():", t32.dtype)

print("t32 * 2 + 1:", t32 * 2 + 1)  # elementwise operations, as in NumPy
result = t32 @ torch.ones(2)  # matrix multiplication, as in NumPy
print("t32 @ ones:", result)

labels = torch.tensor([1.0, 0.0, 1.0])
labels_col = labels.unsqueeze(1)
print("labels:", tuple(labels.shape))
print("after unsqueeze(1):", tuple(labels_col.shape))

Więcej przykładów operacji na tensorach:
[Tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html).

### Autograd

[Autograd](https://docs.pytorch.org/docs/stable/autograd.html) to mechanizm automatycznego różniczkowania w PyTorchu. W trakcie obliczeń PyTorch zapisuje
wszystkie operacje na śledzonych tensorach w **grafie obliczeń**. Jest to skierowany graf acykliczny: liśćmi są tensory wejściowe (np. parametry modelu), a korzeniem -
wynik obliczeń, zwykle koszt. Autograd przechodzi od korzenia
do liści i mnoży pochodne kolejnych operacji zgodnie z regułą łańcuchową - robi więc dokładnie to, co nasza funkcja
`backward` ([graf obliczeń w tutorialu](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#computational-graph),
[Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html#how-autograd-encodes-the-history)).

Na podstawie zdefiniowanych przez nas w kodzie operacji, PyTorch dynamicznie buduje graf obliczeń. Takie podejście jest wygodne i pozwala na łatwe eksperymentowanie, ale odbywa się to potencjalnie kosztem wydajności, ponieważ pozostawia kwestię optymalizacji programiście. Więcej na ten temat dla zainteresowanych na końcu laboratorium. Graf jest budowany od nowa przy każdym przejściu w przód, a po wywołaniu `.backward()` jest domyślnie
zwalniany.

W praktyce wygląda to tak:

1. tensory z `requires_grad=True` są śledzone - parametry warstw mają
   tę flagę ustawioną od razu, a dla własnych tensorów ustawiamy ją sami (dane i etykiety jej nie potrzebują),
2. każda operacja na nich dopisuje węzeł do grafu obliczeń; tensor wynikowy pamięta operację, która go utworzyła,
   w atrybucie `grad_fn` (nie używamy go wprost - korzysta z niego PyTorch),
3. `.backward()` wywołane na koszcie przechodzi graf od końca i mnoży
   pochodne kolejnych operacji (reguła łańcuchowa); pochodne są liczone ze wzorów, więc są dokładne, a nie przybliżone
   jak w `df`,
4. wynik trafia do atrybutu `.grad` każdego śledzonego tensora, który jest
   liściem grafu (np. parametru modelu).

Co dzieje się w środku: [A Gentle Introduction to torch.autograd](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)
oraz [micrograd](https://www.youtube.com/watch?v=VMj-3S1tku0) Andreja Karpathy'ego - autograd napisany od zera.

Sprawdźmy to na funkcji $f(x) = 3x^2 - 4x + 5$ z Zadania 1:

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = 3 * x**2 - 4 * x + 5
print("y:", y)
print("y.grad_fn:", y.grad_fn)  # y remembers the last operation that created it
print("x.grad before backward():", x.grad)

y.backward()
print("x.grad after backward():", x.grad.item())
print("analytical_df(3):       ", analytical_df(3.0))
print("df(f, 3):               ", df(f, 3.0))

PyTorch nie nadpisuje gradientu, tylko **dodaje** nowy do tego, który już jest w `.grad`. Bywa to przydatne, ale w zwykłej
pętli uczącej oznacza, że przed każdym `backward()` trzeba wyzerować gradienty wszystkich parametrów - inaczej w każdej
epoce byłyby coraz większe. Dla pojedynczego tensora służy do tego `.zero_()`:
metody, których nazwa kończy się na `_`, zmieniają tensor w miejscu. Wszystkie parametry modelu naraz zeruje
`optimizer.zero_grad()`.

In [ ]:
y = 3 * x**2 - 4 * x + 5
y.backward()
print("after the second backward():", x.grad.item(), "<- gradients accumulate!")

x.grad.zero_()
print("after x.grad.zero_():", x.grad.item())

Śledzenie operacji kosztuje czas i pamięć, a nie zawsze jest potrzebne, np. przy liczeniu predykcji albo rysowaniu
wykresów. Wyłączamy je blokiem [`with torch.no_grad():`](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html) - operacje wewnątrz nie są
zapisywane w grafie ([więcej w dokumentacji](https://docs.pytorch.org/docs/stable/notes/autograd.html#locally-disabling-gradient-computation)).

`.detach()` zwraca ten sam tensor, ale odcięty od grafu (np. przed `.numpy()`),
a `.item()` zamienia jednoelementowy tensor na liczbę Pythona.

In [ ]:
with torch.no_grad():
    y = 3 * x**2 - 4 * x + 5
print("in torch.no_grad():  requires_grad =", y.requires_grad, " grad_fn =", y.grad_fn)

x_detached = x.detach()
print("after x.detach():    requires_grad =", x_detached.requires_grad)
print("y.item():", y.item(), type(y.item()))

### Modele: `nn.Module`

Model w PyTorchu to klasa dziedzicząca po [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html), podobnie jak nasza
`LogisticRegression` z Zadania 2. Warstwy, np. `nn.Linear`, również dziedziczą po `nn.Module`, więc modele składa się
z mniejszych modułów. Rozmiary wejścia i wyjścia każdej warstwy podajemy jawnie.

| Element | Co robi |
|---|---|
| `__init__` | zaczyna od `super().__init__()`, tworzy warstwy i zapisuje je jako atrybuty - ich parametry ([`nn.Parameter`](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html)) są rejestrowane automatycznie |
| `forward(X)` | przejście w przód; jedyna metoda, którą piszemy |
| `model(X)` | wywołuje `forward` |
| `model.parameters()` | wszystkie parametry modelu; przekazujemy je optymalizatorowi |

Metody `backward` nie piszemy - autograd zbuduje przejście wstecz z operacji wykonanych w `forward`.
Podstawowa warstwa to [`nn.Linear(in_features, out_features)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html),
czyli nasze `X @ w + b`. Zobaczmy, jakie parametry rejestruje
(`named_parameters()` zwraca je razem z nazwami):

In [ ]:
layer = nn.Linear(2, 1)
for name, param in layer.named_parameters():
    print(name, tuple(param.shape), "requires_grad =", param.requires_grad)

print("output for 3 points:", tuple(layer(torch.ones(3, 2)).shape))

#### Zadanie 6 (0.5 punktu)

Uzupełnij `LogisticRegressionTorch` - regresję logistyczną jako `nn.Module`: warstwa liniowa z `n_features` wejściami
i jednym wyjściem (logitem), a po niej sigmoida (`nn.Sigmoid`). Użyj dokładnie nazw atrybutów `self.linear`
i `self.activation`. Zwróć uwagę, że `nn.Linear` przechowuje wagi jako macierz `(out_features, in_features)`, czyli
`(1, 2)` dla naszych danych.

Przykład podobnej klasy: [Build the Neural Network - Define the Class](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html#define-the-class).

In [ ]:
class LogisticRegressionTorch(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        raise NotImplementedError
        self.linear = ...      # nn.Linear: n_features inputs -> 1 output (logit)
        self.activation = ...  # nn.Sigmoid

    def forward(self, X):
        # X: (N, n_features) -> probabilities of class 1, shape (N, 1)
        raise NotImplementedError
        logits = ...  # 1. linear layer applied to X            (N, 1)
        return ...    # 2. activation applied to the logits     (N, 1)

In [ ]:
check_task_6(LogisticRegressionTorch)

### Funkcje kosztu

PyTorch zawiera większość popularnych [funkcji kosztu](https://docs.pytorch.org/docs/stable/nn.html#loss-functions), np. `nn.MSELoss` czy `nn.BCELoss`.
Można ich używać na dwa sposoby, z czego pierwszy jest popularniejszy:
- jako klas z modułu `torch.nn` - tworzymy obiekt, np. `loss_fn = nn.BCELoss()`, i wywołujemy go jak funkcję,
- jako funkcji z modułu [`torch.nn.functional`](https://docs.pytorch.org/docs/stable/nn.functional.html), np. `torch.nn.functional.binary_cross_entropy`.

Funkcja kosztu przyjmuje argumenty w kolejności **(predykcje, etykiety)** - tak samo jak nasza `bce(p_hat, y)` - i zwraca
tensor z jedną liczbą (0-wymiarowy, czyli skalar). Domyślnie jest to średnia po punktach, tak jak w `bce`. Właśnie na
tym skalarze wywołujemy `.backward()` - bez dodatkowych argumentów można to zrobić tylko dla jednej liczby.

In [ ]:
demo_p = torch.tensor([0.9, 0.2, 0.5, 0.99])
demo_y = torch.tensor([1.0, 0.0, 1.0, 0.0])
demo_loss = nn.BCELoss()(demo_p, demo_y)  # order: (predictions, targets)
print("nn.BCELoss:", demo_loss, " shape:", tuple(demo_loss.shape))
print(f"our bce:    {bce(demo_p.numpy(), demo_y.numpy()):.4f}")

Sprawdźmy, że model w PyTorchu liczy te same gradienty co nasza implementacja w NumPy. Kopiujemy parametry z modelu NumPy do modelu PyTorcha i porównujemy
gradient z `loss.backward()` z Twoją funkcją `backward`.

In [ ]:
X_lin_t = torch.tensor(X_lin, dtype=torch.float32)               # (N, 2)
y_lin_t = torch.tensor(y_lin, dtype=torch.float32).unsqueeze(1)  # (N, 1)

np_model = LogisticRegression(2, seed=1)
torch_model = LogisticRegressionTorch(2)
with torch.no_grad():  # copying parameters is not part of the computation graph
    torch_model.linear.weight.copy_(torch.tensor(np_model.w).view(1, 2))
    torch_model.linear.bias.fill_(np_model.b)

loss = nn.BCELoss()(torch_model(X_lin_t), y_lin_t)
loss.backward()
grad_w, grad_b = backward(np_model, X_lin, y_lin)

print(f"cost:   NumPy {bce(np_model.forward(X_lin), y_lin):.6f}   PyTorch {loss.item():.6f}")
print(f"dL/dw:  NumPy {grad_w}   PyTorch {torch_model.linear.weight.grad.numpy().ravel()}")
print(f"dL/db:  NumPy {grad_b:.6f}   PyTorch {torch_model.linear.bias.grad.item():.6f}")

### Optymalizatory

Optymalizator (z modułu[`torch.optim`](https://docs.pytorch.org/docs/stable/optim.html)) wykonuje krok 4 pętli uczącej. Przy tworzeniu dostaje parametry modelu
i stałą uczącą, np. [`optim.SGD`](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html), a potem:

- [`optimizer.zero_grad()`](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html) zeruje gradienty wszystkich tych
  parametrów (domyślnie ustawia `.grad` na `None`, co działa jak zero),
- [`optimizer.step()`](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html) aktualizuje parametry na podstawie gradientów,
  które `loss.backward()` zapisał w `.grad`.

`optim.SGD` wykonuje dokładnie krok z naszej pętli w NumPy: $\theta \leftarrow \theta - \eta \, \nabla_\theta L$. Gradient liczymy
tu na całym zbiorze, więc jest to zwykły spadek wzdłuż gradientu. Trening na losowych paczkach danych i inne
optymalizatory poznamy w dalszej części laboratorium.

Poniżej ten sam spadek wzdłuż gradientu dla funkcji $f$ z Zadania 1 - raz z ręczną aktualizacją,
a raz z optymalizatorem. 

Ręczną aktualizację robimy w `torch.no_grad()`, bo sama zmiana parametru nie powinna trafić do
grafu obliczeń. Optymalizator robi tą samą operację za nas.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
for step in range(50):
    y = 3 * x**2 - 4 * x + 5
    y.backward()
    with torch.no_grad():  # the update itself is not part of the computation graph
        x -= 0.1 * x.grad
    x.grad.zero_()
print(f"by hand:           x = {x.item():.4f}")

x = torch.tensor(3.0, requires_grad=True)
optimizer = optim.SGD([x], lr=0.1)
for step in range(50):
    optimizer.zero_grad()
    y = 3 * x**2 - 4 * x + 5
    y.backward()
    optimizer.step()
print(f"with optimizer:    x = {x.item():.4f}   (minimum: 2/3 = {2 / 3:.4f})")

#### Zadanie 7 (1 punkt)

Uzupełnij `train_torch` - pętlę uczącą w PyTorchu. Optymalizator jest argumentem, więc tej samej funkcji użyjemy
później z innym optymalizatorem. Do historii zapisuj `loss.item()` (zwykła liczba) - sam tensor `loss` trzymałby
w pamięci cały graf obliczeń. Pierwsza linia, `model.train()`, jest podana (wyjaśnienie poniżej).

Przykład podobnej pętli: [Optimizing Model Parameters - Full Implementation](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html#full-implementation);
dokumentacja: [`nn.BCELoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html),
[`torch.optim`](https://docs.pytorch.org/docs/stable/optim.html).

In [ ]:
def train_torch(model, optimizer, X, y, epochs=100):
    # X: (N, d), y: (N, 1), both float32; returns cost history (floats, one per epoch, before the update)
    model.train()  # training mode 
    loss_fn = nn.BCELoss()
    history = []
    for epoch in range(epochs):
        raise NotImplementedError
        ...                  # 1. zero the gradients of all parameters 
        p_hat = ...          # 2. forward pass                  
        loss = ...           # 3. cost
        history.append(...)  #    store the cost as a Python float (.item())
        ...                  # 4. backward pass
        ...                  # 5. update the parameters
    return history

In [ ]:
check_task_7(train_torch, LogisticRegressionTorch)

Do predykcji przełączamy model w tryb ewaluacji ([`model.eval()`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval))
i wyłączamy gradienty (`torch.no_grad()`).
Tryb ma znaczenie dla warstw, które działają inaczej podczas treningu (poznamy je dalej), dlatego pętla ucząca
zaczyna się od [`model.train()`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train). Funkcja `torch_proba` robi to za nas i zwraca wynik jako tablicę NumPy dla naszych
funkcji rysujących.

Wynik jest taki sam jak dla Twojej implementacji w NumPy - krzywe różnią się tylko na starcie, bo modele zaczynają
z innych losowych parametrów.

In [ ]:
def torch_proba(model):
    def predict_proba(X_np):
        model.eval()
        with torch.no_grad():
            return model(torch.tensor(X_np, dtype=torch.float32)).squeeze(1).numpy()
    return predict_proba


torch.manual_seed(0)
logreg_lin = LogisticRegressionTorch(2)
optimizer = optim.SGD(logreg_lin.parameters(), lr=0.5)
history_t = train_torch(logreg_lin, optimizer, X_lin_t, y_lin_t, epochs=100)
plot_training(history_t, torch_proba(logreg_lin), X_lin, y_lin, title="PyTorch", compare={"NumPy": history})

### Zbiór make_moons

[`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) generuje dwa
przeplatające się półkola. Trenujemy na nim tę samą regresję logistyczną (krzyżyki to błędne predykcje).

In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=1000, noise=0.1, random_state=4)
X_moons_t = torch.tensor(X_moons, dtype=torch.float32)               # (N, 2)
y_moons_t = torch.tensor(y_moons, dtype=torch.float32).unsqueeze(1)  # (N, 1)

torch.manual_seed(0)
logreg_moons = LogisticRegressionTorch(2)
optimizer = optim.SGD(logreg_moons.parameters(), lr=0.5)
history_logreg = train_torch(logreg_moons, optimizer, X_moons_t, y_moons_t, epochs=1000)
plot_training(history_logreg, torch_proba(logreg_moons), X_moons, y_moons, title="Logistic regression",
              mark_errors=True)

**Koszt przestaje maleć**, a granica decyzyjna regresji logistycznej to zawsze prosta - dłuższy trening ani inna stała
ucząca tego nie zmienią. Zamiast ręcznie projektować nieliniowe cechy, użyjemy **sieci neuronowej**, która nauczy się ich sama.

---
## Sieć neuronowa MLP

Dlaczego regresja logistyczna nie radzi sobie z `make_moons`? Model przewiduje klasę 1, gdy $\hat p \geq 0.5$.
Sigmoida przekracza $0.5$ dokładnie wtedy, gdy jej argument jest dodatni, więc granicą decyzyjną jest zbiór punktów,
dla których logit wynosi zero:

$$
w_1 x_1 + w_2 x_2 + b = 0
$$

Jest to równanie prostej - niezależnie od wartości wag granica regresji logistycznej zawsze będzie prostą, a trening możeją tylko obracać i przesuwać. Dane z pierwszej części były liniowo separowalne, więc zawsze istniała prosta, która idealnie rozdzielała klasy. Do poradzenia sobie z tym problemem wykorzystamy **sieć neuronową**, która wprowadzi nieliniowość do naszego modelu.

**Neuron** to iloczyn skalarny wag z wejściem, bias i **funkcja aktywacji (activation function)** - nasza regresja
logistyczna to dokładnie jeden neuron z aktywacją sigmoidalną. 

**MLP (multilayer perceptron)** układa neurony w kolejne warstwy. Warstwy między wejściem a wyjściem to **warstwy ukryte (hidden layers)**: ich wyjście $h$ to nowe
cechy (**ukryta reprezentacja**), których sieć uczy się sama. Warstwa wyjściowa to zwykła regresja logistyczna, tyle
że na cechach $h$ zamiast $x$.

Nasza sieć ma warstwę ukrytą z 3 neuronami $\tanh$:

<img src="figures/mlp_architektura.png" width="820" alt="architektura MLP">

$$
h = \tanh(x W_1 + b_1) \qquad \hat p = \sigma(h W_2 + b_2)
$$

gdzie $W_1$ ma kształt $2 \times 3$, a $W_2$ - $3 \times 1$.

Dlaczego sieć poradzi sobie lepiej z `make_moons`? Warstwa wyjściowa nadal wyznacza granicę liniową, ale w przestrzeni
cech $h$, a nie w przestrzeni wejścia $x$. Neurony ukryte $h$ są nieliniowymi funkcjami, więc przekształcają dane tak,
że klasy, których nie da się rozdzielić prostą w przestrzeni $x$, w przestrzeni $h$ można rozdzielić płaszczyzną.

Przeniesiona z powrotem do przestrzeni $x$ granica decyzyjna jest krzywą. Najważniejsze, że sieć uczy się tego
przekształcenia sama, razem z warstwą wyjściową - nie musimy ręcznie projektować nieliniowych cech. Po treningu
zobaczymy to na wykresie przestrzeni ukrytej, a rolę nieliniowej aktywacji $\tanh$ sprawdzisz w pytaniach do Zadania 8.

#### Zadanie 8 (1 punkt)

Uzupełnij `MoonMLP`: w `self.layers` utwórz `nn.Sequential` z warstwami jak na schemacie, a w `forward` wywołaj go.
Przykład użycia `nn.Sequential`: [Build the Neural Network - nn.Sequential](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html#nn-sequential).
Trenujemy Twoją funkcją `train_torch`, ale z optymalizatorem **Adam** (opis poniżej).


Warstwy łączymy
[`nn.Sequential`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html) - nie zwykłą listą
Pythona, bo jej parametry nie trafiłyby do `model.parameters()`.

In [ ]:
class MoonMLP(nn.Module):
    def __init__(self, hidden_size=3):
        super().__init__()
        raise NotImplementedError
        self.layers = nn.Sequential(
            ...,  
            ...,  
            ...,  
            ..., 
        )

    def forward(self, X):
        # X: (N, 2) -> probabilities of class 1, shape (N, 1)
        raise NotImplementedError
        return ...  # pass X through self.layers

In [ ]:
check_task_8(MoonMLP)

Sieć uczymy optymalizatorem **Adam**. Zwykły krok SGD zależy tylko od bieżącego gradientu; dwa popularne ulepszenia:
- **momentum** (`optim.SGD(..., momentum=0.9)`) - krok uwzględnia też kierunek poprzednich kroków, jak kula tocząca
  się z górki; pomaga przejść przez płaskie fragmenty funkcji kosztu,
- **[Adam](https://arxiv.org/abs/1412.6980)** (`optim.Adam`) - momentum + osobna, automatycznie dobierana wielkość
  kroku dla każdego parametru; dobrze działa nawet przy niedokładnie dobranej stałej uczącej i jest domyślnym wyborem
  przy treningu sieci.

In [ ]:
torch.manual_seed(0)  # the network's cost is not convex - the result depends on the random start
mlp = MoonMLP(hidden_size=3)
optimizer = optim.Adam(mlp.parameters(), lr=0.03)
history_mlp = train_torch(mlp, optimizer, X_moons_t, y_moons_t, epochs=1000)

check_mlp_training(mlp, X_moons_t, y_moons_t)
plot_training(history_mlp, torch_proba(mlp), X_moons, y_moons, title="MLP",
              compare={"logistic regression": history_logreg})

Przez pierwsze ok. 200 epok koszt MLP stoi na poziomie regresji logistycznej (sieć zachowuje się jak model liniowy), a potem
gwałtownie spada, gdy sieć „odkrywa”, jak zgiąć granicę. Takie **plateau** są typowe przy uczeniu sieci.
Dokładność liczymy tu na zbiorze treningowym - w praktyce ocenia się model na osobnym zbiorze testowym.

Na plateau dobrze widać różnicę między optymalizatorami. Poniżej trenujemy tę samą sieć zwykłym SGD, SGD z momentum i Adamem, za każdym razem z trzech
różnych losowych startów.

In [ ]:
optimizers = {
    "SGD, lr = 0.1": lambda params: optim.SGD(params, lr=0.1),
    "SGD with momentum, lr = 0.1": lambda params: optim.SGD(params, lr=0.1, momentum=0.9),
    "Adam, lr = 0.03": lambda params: optim.Adam(params, lr=0.03),
}

plt.figure(figsize=(7, 3.8))
for color, (name, make_optimizer) in zip(["C0", "C1", "C2"], optimizers.items()):
    accs = []
    for seed in range(3):  # the same network from three different random starts
        torch.manual_seed(seed)
        net = MoonMLP(hidden_size=3)
        hist = train_torch(net, make_optimizer(net.parameters()), X_moons_t, y_moons_t, epochs=1000)
        plt.plot(hist, color=color, lw=1, label=name if seed == 0 else None)
        accs.append(accuracy(torch_proba(net), X_moons, y_moons))
    print(f"{name}: accuracy {', '.join(f'{acc:.1%}' for acc in accs)}")

plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Same network, different optimizers (3 random starts)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Zwykłe SGD w 1000 epok nie schodzi z plateau i zostaje na poziomie regresji logistycznej. SGD z momentum wychodzi z niego
później i nie przy każdym starcie. Adam wychodzi najwcześniej i za każdym razem.

Zajrzyjmy teraz do środka sieci. Po lewej proste, na których neurony ukryte zmieniają znak - z tych „cięć”
sieć składa granicę. Po prawej punkty w przestrzeni ukrytej $h$: tu klasy rozdziela już **płaszczyzna** ostatniej
warstwy.

In [ ]:
# parameters and hidden representation of the trained network
W_h = mlp.layers[0].weight.detach().numpy()            # (3, 2)
b_h = mlp.layers[0].bias.detach().numpy()              # (3,)
W_out = mlp.layers[2].weight.detach().numpy().ravel()  # (3,)
b_out = mlp.layers[2].bias.item()
with torch.no_grad():
    H = mlp.layers[1](mlp.layers[0](X_moons_t)).numpy()  # (N, 3)

fig = plt.figure(figsize=(10, 4.2))

# left: lines where the hidden neurons change sign
ax1 = fig.add_subplot(1, 2, 1)
plot_decision_boundary(torch_proba(mlp), X_moons, y_moons, ax=ax1, title="Hidden neuron lines")
x1 = np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 50)
for j in range(len(b_h)):
    ax1.plot(x1, -(W_h[j, 0] * x1 + b_h[j]) / W_h[j, 1], "--", lw=2, label=f"neuron $h_{j + 1}$")
ax1.set_ylim(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5)
ax1.legend(loc="lower left", fontsize=8)

# right: points in the hidden space and the plane of the output layer (W_out . h + b_out = 0)
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.scatter(H[:, 0], H[:, 1], H[:, 2], c=y_moons, cmap="coolwarm", s=6, edgecolors="k", linewidths=0.2)
g1, g2 = np.meshgrid(np.linspace(-1, 1, 20), np.linspace(-1, 1, 20))
plane = -(W_out[0] * g1 + W_out[1] * g2 + b_out) / W_out[2]
plane[np.abs(plane) > 1] = np.nan
ax2.plot_surface(g1, g2, plane, alpha=0.3, color="gray")

ax2.set_xlabel("$h_1$")
ax2.set_ylabel("$h_2$")
ax2.set_zlabel("$h_3$")
ax2.set_title("Hidden space")
ax2.view_init(elev=15, azim=-130)  # try other angles, e.g. azim=-60

fig.tight_layout()
plt.show()

Na koniec porównajmy kilka architektur: sieci z 1, 2 i 3 neuronami ukrytymi oraz sieć z 8 neuronami bez
nieliniowości (`nn.Identity()` zamiast `Tanh`).

In [ ]:
def build_and_train(model, epochs=1000):
    optimizer = optim.Adam(model.parameters(), lr=0.03)
    train_torch(model, optimizer, X_moons_t, y_moons_t, epochs=epochs)
    return model


torch.manual_seed(0)
models = {f"{h} neuron(s), tanh": build_and_train(MoonMLP(hidden_size=h)) for h in (1, 2, 3)}
models["8 neurons, no activation"] = build_and_train(
    nn.Sequential(nn.Linear(2, 8), nn.Identity(), nn.Linear(8, 1), nn.Sigmoid()))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, (name, net) in zip(axes, models.items()):
    acc = accuracy(torch_proba(net), X_moons, y_moons)
    plot_decision_boundary(torch_proba(net), X_moons, y_moons, ax=ax, title=f"{name}: {acc:.1%}")
fig.tight_layout()
plt.show()

- Dlaczego sieć z 8 neuronami bez aktywacji wyznacza prostą granicę, mimo że ma więcej neuronów niż pozostałe sieci?
  Wskazówka: podstaw $h = x W_1 + b_1$ do wzoru na logit warstwy wyjściowej.
- Dlaczego granica sieci z jednym neuronem ukrytym $\tanh$ to również prosta?

// skomentuj tutaj